# Case 1 — Data Cleaning and Preprocessing

In this notebook, we perform the data cleaning and preprocessing required before building predictive models. The dataset contains missing values and categorical variables, which must be handled carefully to avoid bias and ensure good generalization performance.

In [22]:
import pandas as pd
import numpy as np

from IPython.display import display

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

## 1/ Load Data

We load the training dataset, which contains both features and the target variable, and the test dataset, which contains only features for which predictions must be made.

In [23]:
train_path = "case1Data.csv"
xnew_path  = "case1Data_Xnew.csv"

train_df = pd.read_csv(train_path)
xnew_df  = pd.read_csv(xnew_path)

print("Train shape:", train_df.shape)
print("Xnew shape:", xnew_df.shape)

display(train_df.head())
display(xnew_df.head())

Train shape: (100, 101)
Xnew shape: (1000, 100)


,y,x_01,x_02,x_03,x_04,x_05,x_06,x_07,x_08,x_09,...,x_91,x_92,x_93,x_94,x_95,C_01,C_02,C_03,C_04,C_05
0,375.823073,6.359019,-13.367120,-2.483750,-6.641891,11.733539,NaN,-17.085361,22.194764,16.827888,...,-10.200888,3.980048,-4.433274,-1.473723,NaN,74.0,72.0,72.0,73.0,73.0
1,266.811730,3.873664,-8.470389,-3.055012,NaN,11.420983,1.822330,-13.694100,22.738654,20.307503,...,-9.740207,NaN,-2.629314,4.816987,-12.240248,74.0,72.0,72.0,73.0,73.0
2,267.271759,5.275824,-12.070531,-1.366168,-4.819100,10.721527,-5.125992,-17.476865,NaN,15.963889,...,-14.501970,10.054005,NaN,NaN,-11.107921,73.0,72.0,75.0,74.0,74.0
3,219.951294,4.430110,-4.467975,-0.730736,-10.047104,11.498539,-2.870260,-14.033012,18.225190,10.409488,...,-11.086963,2.019726,-8.531959,3.520833,NaN,71.0,72.0,73.0,71.0,72.0
4,289.697954,3.116458,-8.518713,-6.796050,NaN,7.646285,-3.118309,-13.102567,22.801217,16.680208,...,-9.117422,6.627601,-2.805531,5.914351,-11.240573,72.0,72.0,72.0,74.0,75.0


,x_01,x_02,x_03,x_04,x_05,x_06,x_07,x_08,x_09,x_10,...,x_91,x_92,x_93,x_94,x_95,C_01,C_02,C_03,C_04,C_05
0,-0.843969,-9.104918,-5.076919,-4.222152,3.606609,-4.505494,-11.481997,16.201722,15.939470,NaN,...,-13.884702,7.465161,-4.667464,3.949705,-10.715577,73.0,72.0,73.0,75.0,73.0
1,0.802093,-10.196678,-4.500370,-7.827837,5.199002,NaN,-15.928708,20.151309,13.707194,-8.517576,...,-14.937164,5.229448,-6.927970,3.271193,-12.420893,73.0,72.0,73.0,71.0,75.0
2,4.234883,-10.798261,-0.465914,-6.054850,NaN,NaN,-16.182312,16.419564,12.152861,-6.418069,...,-11.058964,4.692879,-0.929818,NaN,-14.551448,NaN,72.0,73.0,71.0,73.0
3,7.041336,-5.169413,-4.158334,-4.270638,14.939894,0.008338,-10.556799,NaN,14.180830,NaN,...,NaN,7.460901,-2.484389,8.149697,-11.598544,72.0,72.0,74.0,75.0,75.0
4,1.135564,-12.048088,-4.828939,-6.565217,7.493100,-2.789944,-15.859234,21.560086,14.147759,-4.848519,...,-9.607803,5.654679,-3.020357,3.030958,-13.320599,72.0,72.0,71.0,74.0,73.0


We immediately inspect the first rows to understand the structure and detect that `C_01`…`C_05` are categorical factors encoded as numeric codes.

## 2/ Separate target and features

We separate the response variable `y` from the feature matrix `X`.

In [24]:
y = train_df["y"].copy()
X = train_df.drop(columns=["y"]).copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (100, 100)
y shape: (100,)


## 3/ Identify variable types

We identify column types: numerical or cateforical.

In [26]:
# Numerical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Categorical columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print("Numerical columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))

Numerical columns: 100
Categorical columns: 0


## 4/ Correcting C_01…C_05

Since `C_01`…`C_05` are categorical factors (even if stored as numbers), we cast them to categorical variables.  
This ensures they are treated with one-hot encoding rather than as continuous numerical inputs.

In [27]:
forced_cat = ["C_01", "C_02", "C_03", "C_04", "C_05"]

# Safety checks
missing_in_X = [c for c in forced_cat if c not in X.columns]
missing_in_xnew = [c for c in forced_cat if c not in xnew_df.columns]
if missing_in_X:
    raise ValueError(f"Missing expected columns in training features X: {missing_in_X}")
if missing_in_xnew:
    raise ValueError(f"Missing expected columns in Xnew: {missing_in_xnew}")

# Convert C_01..C_05 to object strings, and turn <NA> into np.nan
for c in forced_cat:
    X[c] = X[c].astype("string").replace({pd.NA: np.nan}).astype("object")
    xnew_df[c] = xnew_df[c].astype("string").replace({pd.NA: np.nan}).astype("object")

# Define columns (exactly these are categorical)
cat_cols = forced_cat[:]
num_cols = [c for c in X.columns if c not in cat_cols]

print("Numerical columns:", len(num_cols))
print("Categorical columns:", len(cat_cols), cat_cols)

Numerical columns: 95
Categorical columns: 5 ['C_01', 'C_02', 'C_03', 'C_04', 'C_05']


## 5/ Define Preprocessing

We define two preprocessing pipelines:

- Numerical variables: missing values are imputed using the median.
- Categorical variables: missing values are imputed using the most frequent category and encoded using one-hot encoding.

Before applying the pipeline, we ensure that all missing values are represented as `np.nan` to avoid compatibility issues with scikit-learn.

FIX: ensure no pd.NA remains

In [39]:
# Convert entire dataset to use np.nan instead of pd.NA
X = X.astype("object").where(pd.notna(X), np.nan)
xnew_df = xnew_df.astype("object").where(pd.notna(xnew_df), np.nan)

# Ensure categorical columns are object (safe for sklearn)
forced_cat = ["C_01", "C_02", "C_03", "C_04", "C_05"]
for c in forced_cat:
    X[c] = X[c].astype("object")
    xnew_df[c] = xnew_df[c].astype("object")

Pipelines

In [38]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols),
    ],
    remainder="drop"
)

## 6/ Fit on Training Data and Transform Both Datasets

The preprocessing pipeline is fitted only on the training data to avoid data leakage.  
The same transformations are then applied to both the training and test sets.

In [33]:
X_clean = preprocessor.fit_transform(X)
Xnew_clean = preprocessor.transform(xnew_df)

print("Processed train shape:", X_clean.shape)
print("Processed test shape:", Xnew_clean.shape)

Processed train shape: (100, 116)
Processed test shape: (1000, 116)


## 7/ Convert to dataframe

We recover feature names (numerical + one-hot) and convert the processed matrices into DataFrames for inspection and later modeling.

In [35]:
# Feature names: numeric + OHE
feature_names = []
feature_names.extend(num_cols)

ohe = preprocessor.named_transformers_["cat"].named_steps["encoder"]
feature_names.extend(list(ohe.get_feature_names_out(cat_cols)))

# Convert to DataFrame (dense for readability; still fine for this dataset size)
X_clean_df = pd.DataFrame(
    X_clean.toarray() if hasattr(X_clean, "toarray") else X_clean,
    columns=feature_names
)
Xnew_clean_df = pd.DataFrame(
    Xnew_clean.toarray() if hasattr(Xnew_clean, "toarray") else Xnew_clean,
    columns=feature_names
)

display(X_clean_df.head())

,x_01,x_02,x_03,x_04,x_05,x_06,x_07,x_08,x_09,x_10,...,C_04_71.0,C_04_72.0,C_04_73.0,C_04_74.0,C_04_75.0,C_05_71.0,C_05_72.0,C_05_73.0,C_05_74.0,C_05_75.0
0,6.359019,-13.367120,-2.483750,-6.641891,11.733539,-3.771067,-17.085361,22.194764,16.827888,-10.367142,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,3.873664,-8.470389,-3.055012,-5.793856,11.420983,1.822330,-13.694100,22.738654,20.307503,-2.859097,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,5.275824,-12.070531,-1.366168,-4.819100,10.721527,-5.125992,-17.476865,22.435510,15.963889,-3.257940,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,4.430110,-4.467975,-0.730736,-10.047104,11.498539,-2.870260,-14.033012,18.225190,10.409488,-5.616061,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,3.116458,-8.518713,-6.796050,-5.793856,7.646285,-3.118309,-13.102567,22.801217,16.680208,-3.357765,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


## 8/ Save clean data

We save the cleaned datasets for model training and prediction.

In [40]:
X_clean_df.to_csv("X_clean.csv", index=False)
Xnew_clean_df.to_csv("Xnew_clean.csv", index=False)
pd.DataFrame({"y": y}).to_csv("y.csv", index=False)

print("Saved: X_clean.csv, Xnew_clean.csv, y.csv")

Saved: X_clean.csv, Xnew_clean.csv, y.csv
